In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 25


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.367602925747633
Epoch 2/100, Loss: 1.2885583154857159
Epoch 3/100, Loss: 1.377865243703127
Epoch 4/100, Loss: 1.311681054532528
Epoch 5/100, Loss: 1.3777991272509098
Epoch 6/100, Loss: 1.309329330921173
Epoch 7/100, Loss: 1.422533992677927
Epoch 8/100, Loss: 1.3402817845344543
Epoch 9/100, Loss: 1.33986447006464
Epoch 10/100, Loss: 1.433325245976448
Epoch 11/100, Loss: 1.3501390740275383
Epoch 12/100, Loss: 1.3180335722863674
Epoch 13/100, Loss: 1.3431347236037254
Epoch 14/100, Loss: 1.4492721036076546
Epoch 15/100, Loss: 1.3575992435216904


Epoch 16/100, Loss: 1.4017279222607613
Epoch 17/100, Loss: 1.4256810694932938
Epoch 18/100, Loss: 1.2631438560783863
Epoch 19/100, Loss: 1.5494520030915737
Epoch 20/100, Loss: 1.2646828815340996
Epoch 21/100, Loss: 1.397046908736229
Epoch 22/100, Loss: 1.4555854760110378
Epoch 23/100, Loss: 1.3111934401094913
Epoch 24/100, Loss: 1.3471871726214886
Epoch 25/100, Loss: 1.333457324653864
Epoch 26/100, Loss: 1.66649279743433
Epoch 27/100, Loss: 1.368477936834097
Epoch 28/100, Loss: 1.4536998346447945
Epoch 29/100, Loss: 1.4392304867506027
Epoch 30/100, Loss: 1.521172922104597


Epoch 31/100, Loss: 1.3879291117191315
Epoch 32/100, Loss: 1.2425982765853405
Epoch 33/100, Loss: 1.3890261799097061
Epoch 34/100, Loss: 1.4316660948097706
Epoch 35/100, Loss: 1.403300128877163
Epoch 36/100, Loss: 1.3394816033542156
Epoch 37/100, Loss: 1.3855503611266613
Epoch 38/100, Loss: 1.366177886724472
Epoch 39/100, Loss: 1.4672570265829563
Epoch 40/100, Loss: 1.306336011737585
Epoch 41/100, Loss: 1.355732899159193
Epoch 42/100, Loss: 1.4731609672307968
Epoch 43/100, Loss: 1.4046719260513783
Epoch 44/100, Loss: 1.3692686781287193
Epoch 45/100, Loss: 1.4137968011200428


Epoch 46/100, Loss: 1.3839510455727577
Epoch 47/100, Loss: 1.4672730825841427
Epoch 48/100, Loss: 1.335895575582981
Epoch 49/100, Loss: 1.2664295583963394
Epoch 50/100, Loss: 1.4403954446315765
Epoch 51/100, Loss: 1.303394541144371
Epoch 52/100, Loss: 1.2466616779565811
Epoch 53/100, Loss: 1.3344763405621052
Epoch 54/100, Loss: 1.35344710201025
Epoch 55/100, Loss: 1.6978360824286938
Epoch 56/100, Loss: 1.3640297651290894
Epoch 57/100, Loss: 1.3262880332767963
Epoch 58/100, Loss: 1.6779332049190998
Epoch 59/100, Loss: 1.2930389568209648
Epoch 60/100, Loss: 1.347729541361332


Epoch 61/100, Loss: 1.3659018501639366
Epoch 62/100, Loss: 1.394734624773264
Epoch 63/100, Loss: 1.340853314846754
Epoch 64/100, Loss: 1.346323885023594
Epoch 65/100, Loss: 1.3249513618648052
Epoch 66/100, Loss: 1.437639456242323
Epoch 67/100, Loss: 1.1850440837442875
Epoch 68/100, Loss: 1.3097799383103848
Epoch 69/100, Loss: 1.3094795569777489
Epoch 70/100, Loss: 1.6463765650987625
Epoch 71/100, Loss: 1.3481537401676178
Epoch 72/100, Loss: 1.3559296019375324
Epoch 73/100, Loss: 1.354762002825737


...

Epoch 25/100, Loss: 1.9141515716910362
Epoch 26/100, Loss: 2.0978480726480484
Epoch 27/100, Loss: 1.9441277012228966
Epoch 28/100, Loss: 1.9846079647541046
Epoch 29/100, Loss: 2.1225070506334305
Epoch 30/100, Loss: 2.0209924653172493
Epoch 31/100, Loss: 1.8994956612586975
Epoch 32/100, Loss: 1.8645075261592865
Epoch 33/100, Loss: 2.2497868463397026
Epoch 34/100, Loss: 2.11280570179224
Epoch 35/100, Loss: 1.9618922844529152
Epoch 36/100, Loss: 2.0572793558239937
Epoch 37/100, Loss: 2.1542331501841545
Epoch 38/100, Loss: 2.1617360189557076
Epoch 39/100, Loss: 2.1159436479210854
Epoch 40/100, Loss: 2.113968387246132
Epoch 41/100, Loss: 2.092355765402317
Epoch 42/100, Loss: 2.1974405720829964


Epoch 43/100, Loss: 2.1553621739149094
Epoch 44/100, Loss: 2.1378308907151222
Epoch 45/100, Loss: 2.072175994515419
Epoch 46/100, Loss: 2.2570207938551903
Epoch 47/100, Loss: 2.2106983363628387
Epoch 48/100, Loss: 2.0990361720323563
Epoch 49/100, Loss: 2.0647548139095306
Epoch 50/100, Loss: 2.026636689901352
Epoch 51/100, Loss: 2.090229742228985
Epoch 52/100, Loss: 2.003439776599407
Epoch 53/100, Loss: 2.051235370337963
Epoch 54/100, Loss: 1.9544221386313438
Epoch 55/100, Loss: 2.0659151524305344
Epoch 56/100, Loss: 2.0225946828722954


Epoch 57/100, Loss: 1.8334097787737846
Epoch 58/100, Loss: 2.102478176355362
Epoch 59/100, Loss: 2.287470407783985
Epoch 60/100, Loss: 2.684462547302246
Epoch 61/100, Loss: 2.0252553522586823
Epoch 62/100, Loss: 2.0767867118120193
Epoch 63/100, Loss: 2.035148397088051
Epoch 64/100, Loss: 2.0821004658937454
Epoch 65/100, Loss: 2.079880081117153
Epoch 66/100, Loss: 2.0322166457772255
Epoch 67/100, Loss: 1.9644887447357178
Epoch 68/100, Loss: 1.888758584856987
Epoch 69/100, Loss: 2.096751593053341
Epoch 70/100, Loss: 2.1417941078543663
Epoch 71/100, Loss: 2.2036653831601143
Epoch 72/100, Loss: 2.162134572863579


Epoch 73/100, Loss: 2.1017395108938217
Epoch 74/100, Loss: 1.9284226298332214
Epoch 75/100, Loss: 1.9798932000994682
Epoch 76/100, Loss: 2.072380229830742
Epoch 77/100, Loss: 2.1268218606710434
Epoch 78/100, Loss: 2.0290692895650864
Epoch 79/100, Loss: 2.189333952963352
Epoch 80/100, Loss: 2.136449307203293
Epoch 81/100, Loss: 2.0402229130268097
Epoch 82/100, Loss: 2.066027410328388
Epoch 83/100, Loss: 1.8633805885910988
Epoch 84/100, Loss: 2.129885397851467
Epoch 85/100, Loss: 1.9193779341876507
Epoch 86/100, Loss: 2.1367395222187042
Epoch 87/100, Loss: 2.114750474691391
Epoch 88/100, Loss: 2.1284139901399612
Epoch 89/100, Loss: 2.0094793513417244


Epoch 90/100, Loss: 1.8666162192821503
Epoch 91/100, Loss: 2.069741301238537
Epoch 92/100, Loss: 1.9821178391575813
Epoch 93/100, Loss: 2.013818956911564
Epoch 94/100, Loss: 2.156256467103958
Epoch 95/100, Loss: 2.116410069167614
Epoch 96/100, Loss: 2.0543883442878723
Epoch 97/100, Loss: 2.0056143701076508
Epoch 98/100, Loss: 2.2050529047846794
Epoch 99/100, Loss: 2.1930426880717278
Epoch 100/100, Loss: 2.028275929391384
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5037
